# 01 - Big Data Storage and Processing

CSE-CIC-IDS2018, 16,233,002 flows, 6.5 GB across 10 daily CSVs.

Builds a bronze/silver/gold lakehouse and takes the storage and processing
measurements. Model training is in notebook 02.

Written as an iterative record - what was tried, what broke, what the numbers say.

## 0. Setup

In [ ]:
import json, os, shutil, time, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings("ignore")

PROJECT = Path(r"D:\Saad A2")
DATA    = PROJECT / "datasets"
LAKE    = PROJECT / "lakehouse"
BRONZE  = LAKE / "bronze"
SILVER  = LAKE / "silver"
GOLD    = LAKE / "gold"
RESULTS = PROJECT / "results"
for p in (LAKE, RESULTS):
    p.mkdir(exist_ok=True)

# 9 files have 80 cols, this one has 84 (see section 2). Typo is the dataset's own.
WIDE_FILE   = "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv"
WIDE_EXTRAS = ["Flow ID", "Src IP", "Src Port", "Dst IP"]

# dropped under P2/P3, kept under P1 so the effect gets measured
LEAKY = ["Dst Port", "Timestamp"] + WIDE_EXTRAS

BENIGN    = "Benign"
ZERO_DAY  = "Bot"                      # held out of training under P3
TS_FORMAT = "dd/MM/yyyy HH:mm:ss"      # day first, not US format

CSV_FILES = sorted(DATA.glob("*.csv"))
print(f"{len(CSV_FILES)} files, {sum(f.stat().st_size for f in CSV_FILES)/1024**3:.2f} GB")
for f in CSV_FILES:
    print(f"  {f.stat().st_size/1024**2:8.1f} MB  {f.name}")

10 files, 6.41 GB
     336.0 MB  Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
     318.3 MB  Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
     365.1 MB  Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
    3867.1 MB  Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
     102.8 MB  Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
     358.5 MB  Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
     364.9 MB  Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
     341.6 MB  Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
     313.7 MB  Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
     199.6 MB  Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


## 1. Iteration 0 - single node baseline

Need to show pandas actually can't handle this, not just claim it.
Load one file, measure, extrapolate, then try the full load.

Spark is deliberately NOT started yet. The JVM would reserve 8 GB up front and
the pandas test would then be competing against it, which would not be a fair
measurement. Restart the kernel before running this section if Spark has
already been started in this session.

In [ ]:
# 1.0 fair-test check. This measurement is only meaningful if the machine is
# not already loaded. Close other apps if free memory is low.
import psutil
free0  = psutil.virtual_memory().available / 1024**3
total0 = psutil.virtual_memory().total / 1024**3
print(f"RAM {total0:.1f} GB total, {free0:.1f} GB free ({100*free0/total0:.0f}%)")
if free0 < 2.5:
    print("\nWARNING: less than half of RAM is free.")
    print("The baseline below will understate what pandas could do on an idle machine.")
    print("Restart the kernel, close other applications, and re-run this section.")
else:
    print("ok - enough headroom for a fair baseline")

RAM 7.9 GB total, 3.4 GB free (43%)
ok - enough headroom for a fair baseline


In [ ]:
# 1.1 one file
single = DATA / "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv"   # smallest, 103 MB

t0 = time.perf_counter()
df_single = pd.read_csv(single, low_memory=False)
t_single = time.perf_counter() - t0

mem_single  = df_single.memory_usage(deep=True).sum() / 1024**3
disk_single = single.stat().st_size / 1024**3

print(f"on disk       : {disk_single:.2f} GB")
print(f"rows x cols   : {df_single.shape[0]:,} x {df_single.shape[1]}")
print(f"load time     : {t_single:.1f} s")
print(f"in memory     : {mem_single:.2f} GB")
print(f"multiplier    : {mem_single/disk_single:.1f}x disk")

on disk       : 0.10 GB
rows x cols   : 331,125 x 80
load time     : 8.8 s
in memory     : 1.48 GB
multiplier    : 14.8x disk


In [ ]:
# 1.2 extrapolate to all 10
disk_total     = sum(f.stat().st_size for f in CSV_FILES) / 1024**3
projected_mem  = disk_total * (mem_single / disk_single)
projected_time = t_single * (disk_total / disk_single)
available      = psutil.virtual_memory().available / 1024**3
total_ram      = psutil.virtual_memory().total / 1024**3

print(f"corpus on disk      : {disk_total:.2f} GB")
print(f"projected memory    : {projected_mem:.1f} GB")
print(f"projected load time : {projected_time/60:.1f} min")
print(f"machine RAM         : {total_ram:.1f} GB  ({available:.1f} GB free)")
print(f"verdict: {'exceeds available memory' if projected_mem > available else 'fits, check timing'}")

legacy_projection = dict(
    disk_gb=round(disk_total, 2),
    projected_mem_gb=round(projected_mem, 1),
    projected_load_min=round(projected_time / 60, 1),
    machine_ram_gb=round(total_ram, 1),
    available_gb=round(available, 1),
    single_file_rows=int(df_single.shape[0]),
    memory_multiplier=round(mem_single / disk_single, 1),
)
del df_single

corpus on disk      : 6.41 GB
projected memory    : 94.6 GB
projected load time : 9.3 min
machine RAM         : 7.9 GB  (3.2 GB free)
verdict: exceeds available memory


### 1.3 Full load attempt

Loads in chunks rather than whole files, so the result is the actual row ceiling
rather than just "could not start". Stops before the machine begins swapping.

In [ ]:
# 1.3 chunked load until memory runs out
SAFETY_GB  = 0.7          # stop before the OS starts swapping
CHUNK_ROWS = 100_000

frames, loaded_rows, stopped_at = [], 0, None
t0 = time.perf_counter()

for i, f in enumerate(CSV_FILES, 1):
    if stopped_at:
        break
    for chunk in pd.read_csv(f, low_memory=False, chunksize=CHUNK_ROWS):
        free = psutil.virtual_memory().available / 1024**3
        if free < SAFETY_GB:
            stopped_at = f"{f.name} after {loaded_rows:,} rows"
            print(f"\nout of memory in file {i}/{len(CSV_FILES)} ({f.name})")
            print(f"  {free:.2f} GB free, floor {SAFETY_GB} GB")
            break
        try:
            frames.append(chunk)
            loaded_rows += len(chunk)
        except MemoryError:
            stopped_at = f"{f.name} MemoryError at {loaded_rows:,} rows"
            break
    print(f"  [{i:2}/{len(CSV_FILES)}] {f.name[:36]:36} "
          f"cumulative {loaded_rows:>10,} rows | "
          f"{psutil.virtual_memory().available/1024**3:5.2f} GB free")

t_legacy = time.perf_counter() - t0
pct = 100 * loaded_rows / 16_233_002
print(f"\npandas held {loaded_rows:,} of 16,233,002 rows ({pct:.1f}%) "
      f"in {t_legacy/60:.1f} min")
print(f"stopped at: {stopped_at or 'completed'}")

legacy_projection.update(
    rows_loaded_single_node=int(loaded_rows),
    pct_of_corpus=round(pct, 1),
    stopped_at=stopped_at,
    elapsed_min=round(t_legacy / 60, 1),
    chunk_rows=CHUNK_ROWS,
)
del frames
import gc; gc.collect()
print(f"memory released, {psutil.virtual_memory().available/1024**3:.2f} GB free")

  [ 1/10] Friday-02-03-2018_TrafficForML_CICFl cumulative  1,048,575 rows |  2.92 GB free
  [ 2/10] Friday-16-02-2018_TrafficForML_CICFl cumulative  2,097,150 rows |  2.38 GB free
  [ 3/10] Friday-23-02-2018_TrafficForML_CICFl cumulative  3,145,725 rows |  1.78 GB free

out of memory in file 4/10 (Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv)
  0.66 GB free, floor 0.7 GB
  [ 4/10] Thuesday-20-02-2018_TrafficForML_CIC cumulative  5,245,725 rows |  0.66 GB free

pandas held 5,245,725 of 16,233,002 rows (32.3%) in 2.0 min
stopped at: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv after 5,245,725 rows
memory released, 4.06 GB free


Now start Spark. Everything from here runs on the distributed engine.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("CA1-BigData-ETL")
    .master("local[*]")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version}, {spark.sparkContext.defaultParallelism} cores")
print(f"driver memory: {spark.conf.get('spark.driver.memory')}")

Spark 3.5.9, 4 cores
driver memory: 3g


## 2. Iteration 1 - naive read, and why it is wrong

Obvious approach is to point Spark at the folder. It runs without error
and the result is wrong.

In [ ]:
# 2.1 naive folder read
naive = spark.read.csv(str(DATA / "*.csv"), header=True, inferSchema=False)
print(f"columns  : {len(naive.columns)}")
print(f"first 6  : {naive.columns[:6]}")
print(f"rows     : {naive.count():,}")
print("no error raised")

columns  : 80
first 6  : ['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts']
rows     : 16,233,002
no error raised


In [ ]:
# 2.1 naive folder read
naive = spark.read.csv(str(DATA / "*.csv"), header=True, inferSchema=False)
print(f"columns  : {len(naive.columns)}")
print(f"first 6  : {naive.columns[:6]}")
print(f"rows     : {naive.count():,}")
print("no error raised")

columns  : 80
first 6  : ['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts']
rows     : 16,233,002
no error raised


The wide file has 4 extra leading columns, so a positional union shifts every
field by 4. Nothing errors, row count is right, schema looks fine, and all
downstream stats are computed on misaligned data.

This is the first of two silent defects here.

In [ ]:
# 2.3 show the corruption. Protocol should only be 6, 17 or 0.
print("Protocol under the naive read:")
naive.groupBy("Protocol").count().orderBy(F.desc("count")).show(8, truncate=False)

print("Protocol when the wide file is read on its own:")
(spark.read.csv(str(DATA / WIDE_FILE), header=True)
      .groupBy("Protocol").count().orderBy(F.desc("count")).show(8, truncate=False))

Protocol under the naive read:
+--------------+-------+
|Protocol      |count  |
+--------------+-------+
|6             |6504691|
|17            |1685323|
|0             |94181  |
|8.6.0.1       |93775  |
|18.219.9.1    |58532  |
|18.218.229.235|58398  |
|52.14.136.135 |58137  |
|18.218.55.126 |57998  |
+--------------+-------+
only showing top 8 rows

Protocol when the wide file is read on its own:
+--------+-------+
|Protocol|count  |
+--------+-------+
|6       |5295644|
|17      |2508972|
|0       |144132 |
+--------+-------+



## 3. Bronze

Data as captured. Only fix is the D2 schema mismatch so the files can be
unioned truthfully. No cleaning yet - that goes in silver where it can be counted.

In [ ]:
# 3.1 read each schema separately, align, union
t0 = time.perf_counter()

narrow_files = [str(f) for f in CSV_FILES if f.name != WIDE_FILE]
df_narrow = spark.read.csv(narrow_files, header=True, inferSchema=False)
df_wide   = (spark.read.csv(str(DATA / WIDE_FILE), header=True, inferSchema=False)
                  .drop(*WIDE_EXTRAS))

assert df_narrow.columns == df_wide.columns, "alignment failed"
print(f"aligned on {len(df_narrow.columns)} columns")

bronze = df_narrow.unionByName(df_wide)        # by name, not position
bronze_rows = bronze.count()
print(f"bronze rows: {bronze_rows:,} (expected 16,233,002)")
print(f"took {time.perf_counter()-t0:.1f} s")

aligned on 80 columns
bronze rows: 16,233,002 (expected 16,233,002)
took 311.9 s


In [ ]:
# 3.2 write bronze
t0 = time.perf_counter()
bronze.write.mode("overwrite").parquet(str(BRONZE))
t_bronze_write = time.perf_counter() - t0

bronze_bytes = sum(f.stat().st_size for f in BRONZE.rglob("*.parquet"))
print(f"written in {t_bronze_write:.1f} s, {bronze_bytes/1024**3:.2f} GB")

written in 441.4 s, 2.39 GB


## 4. Iteration 2 - Silver

Each defect counted before it is fixed, so the count is evidence.

Order matters: D1 before timestamp parsing (header rows hold the literal string
"Timestamp"), D2 before the union.

In [ ]:
# 4.1 D1 - header rows inside the data
bronze = spark.read.parquet(str(BRONZE))
d1_count = bronze.filter(F.col("Label") == "Label").count()
print(f"D1 header rows in bronze: {d1_count}\n")

# attribute to source files - must read CSVs, input_file_name on parquet
# would return parquet paths
print("per file:")
for f in CSV_FILES:
    n = (spark.read.csv(str(f), header=True, inferSchema=False)
              .filter(F.col("Label") == "Label").count())
    if n:
        print(f"    {f.name[:48]:48} {n:>3}")

silver = bronze.filter(F.col("Label") != "Label")
print(f"\nrows after D1 removal: {silver.count():,}")

D1 header rows in bronze: 59

per file:
    Friday-16-02-2018_TrafficForML_CICFlowMeter.csv    1
    Thursday-01-03-2018_TrafficForML_CICFlowMeter.cs  25
    Wednesday-28-02-2018_TrafficForML_CICFlowMeter.c  33

rows after D1 removal: 16,232,943


In [ ]:
# 4.2 D3 - label spellings
labels = silver.groupBy("Label").count().orderBy(F.desc("count")).collect()
for r in labels:
    print(f"  {r['count']:>10,}  {r['Label']}")
print(f"\n{len(labels)} classes")
print("Infilteration is the dataset's own misspelling. Kept verbatim so results")
print("stay traceable to the source files.")

  13,484,708  Benign
     686,012  DDOS attack-HOIC
     576,191  DDoS attacks-LOIC-HTTP
     461,912  DoS attacks-Hulk
     286,191  Bot
     193,360  FTP-BruteForce
     187,589  SSH-Bruteforce
     161,934  Infilteration
     139,890  DoS attacks-SlowHTTPTest
      41,508  DoS attacks-GoldenEye
      10,990  DoS attacks-Slowloris
       1,730  DDOS attack-LOIC-UDP
         611  Brute Force -Web
         230  Brute Force -XSS
          87  SQL Injection

15 classes
Infilteration is the dataset's own misspelling. Kept verbatim so results
stay traceable to the source files.


### 4.3 Timestamp parsing

Timestamps are day first. Parsing as US month-first does not fail - for the two
March files the day is under 13, so MM/dd gives a valid but wrong date and moves
them from last to first in time order. Those days hold Infilteration and all of
Bot, which is the zero-day holdout.

In [ ]:
# 4.3 timestamps, the format trap, and defect D7a
trap = (silver.limit(20000).select(
            F.to_timestamp("Timestamp", "dd/MM/yyyy HH:mm:ss").alias("day_first"),
            F.to_timestamp("Timestamp", "MM/dd/yyyy HH:mm:ss").alias("month_first")))
t_ok, t_bad = trap.agg(F.count("day_first"), F.count("month_first")).first()
print(f"20,000 rows: dd/MM parsed {t_ok:,}, MM/dd parsed {t_bad:,}")
print("both parse, so the wrong format fails silently\n")

silver = (silver.withColumn("ts", F.to_timestamp("Timestamp", TS_FORMAT))
                .withColumn("capture_date", F.to_date("ts")))

bad_ts = silver.filter(F.col("ts").isNull()).count()
lo_raw, hi_raw = silver.agg(F.min("ts"), F.max("ts")).first()
print(f"unparseable: {bad_ts}")
print(f"raw range  : {lo_raw} to {hi_raw}")

# D7a - records outside the capture window.
# The capture ran 14 Feb to 2 Mar 2018. Anything outside that is corrupt.
# These parse fine, so they would pass unnoticed without an explicit check.
WINDOW_LO, WINDOW_HI = "2018-02-14", "2018-03-03"
out_of_window = silver.filter((F.col("ts") < F.lit(WINDOW_LO)) |
                              (F.col("ts") >= F.lit(WINDOW_HI)))
d7a = out_of_window.count()

if d7a:
    print(f"\nD7a - {d7a} rows outside the capture window:")
    out_of_window.select("Timestamp", "Dst Port", "Protocol",
                         "Flow Duration", "Label").show(5, truncate=False)
    neg = out_of_window.filter(F.col("Flow Duration").cast("double") < 0).count()
    print(f"  of these, {neg}/{d7a} also have negative Flow Duration")
    print(f"  labels: {[r['Label'] for r in out_of_window.groupBy('Label').count().collect()]}")
    print("  -> corrupt CICFlowMeter records. Negative duration underflows the")
    print("     timestamp back to 1970. Removing them.")
    silver = silver.filter((F.col("ts") >= F.lit(WINDOW_LO)) &
                           (F.col("ts") < F.lit(WINDOW_HI)))

lo, hi = silver.agg(F.min("ts"), F.max("ts")).first()
print(f"\nclean range: {lo} to {hi}")

assert bad_ts == 0, f"{bad_ts:,} timestamps failed to parse"
assert str(lo)[:10] == "2018-02-14" and str(hi)[:10] == "2018-03-02", (
    f"range is {str(lo)[:10]} to {str(hi)[:10]}, expected 2018-02-14 to 2018-03-02. "
    "Either the dd/MM vs MM/dd trap fired or the session timezone is shifting dates.")
print("assertions passed")

20,000 rows: dd/MM parsed 20,000, MM/dd parsed 0
both parse, so the wrong format fails silently

unparseable: 0
raw range  : 1970-01-05 08:01:17 to 2018-03-02 17:59:59

D7a - 14 rows outside the capture window:
+-------------------+--------+--------+-------------+------+
|Timestamp          |Dst Port|Protocol|Flow Duration|Label |
+-------------------+--------+--------+-------------+------+
|10/01/1970 03:04:26|0       |0       |-188505000000|Benign|
|11/01/1970 12:05:36|0       |0       |-74877000000 |Benign|
|11/01/1970 05:12:30|0       |0       |-4834000000  |Benign|
|11/01/1970 03:51:32|0       |0       |-828220000000|Benign|
|12/01/1970 06:40:49|0       |0       |-578768000000|Benign|
+-------------------+--------+--------+-------------+------+
only showing top 5 rows

  of these, 14/14 also have negative Flow Duration
  labels: ['Benign']
  -> corrupt CICFlowMeter records. Negative duration underflows the
     timestamp back to 1970. Removing them.

clean range: 2018-02-14 06:00:

In [ ]:
# 4.4 cast to double, count D6 (Inf/NaN) and D7 (negative durations)
meta_cols    = ["Timestamp", "ts", "capture_date", "Label"]
feature_cols = [c for c in silver.columns if c not in meta_cols]

for c in feature_cols:
    silver = silver.withColumn(c, F.col(c).cast("double"))

inf_expr = [F.sum(F.when(F.col(c).isin(float("inf"), float("-inf"))
                         | F.isnan(F.col(c)) | F.col(c).isNull(), 1)
                   .otherwise(0)).alias(c) for c in feature_cols]
d6 = {k: v for k, v in silver.agg(*inf_expr).first().asDict().items() if v}

print("D6 columns with Inf/NaN/null:")
for k, v in sorted(d6.items(), key=lambda x: -x[1]):
    print(f"  {k:28} {v:>10,}")

d7 = silver.filter(F.col("Flow Duration") < 0).count()
print(f"\nD7 negative Flow Duration (after D7a removal): {d7:,}")
print("  all 14 negative-duration rows were the D7a records removed in 4.3")

D6 columns with Inf/NaN/null:
  Flow Byts/s                      95,760
  Flow Pkts/s                      95,760

D7 negative Flow Duration (after D7a removal): 0
  all 14 negative-duration rows were the D7a records removed in 4.3


In [ ]:
# 4.5 fix D6, count D5 duplicates, write silver
for c in d6:
    silver = silver.withColumn(
        c, F.when(F.col(c).isin(float("inf"), float("-inf")) | F.isnan(F.col(c)),
                  None).otherwise(F.col(c)))

# D5 duplicates. dropDuplicates on 79 columns shuffles every column and will
# not fit in a 3g driver, so hash each row to one 64-bit value first and
# shuffle that instead. Same answer, far less data moved.
rows_before = silver.count()
row_hash = F.xxhash64(*[F.col(c) for c in feature_cols + ["Label"]])
distinct_rows = (silver.select(row_hash.alias("h"))
                       .agg(F.countDistinct("h")).first()[0])
d5 = rows_before - distinct_rows
print(f"D5 duplicates: {d5:,} ({100*d5/rows_before:.2f}%)")

t0 = time.perf_counter()
(silver.write.mode("overwrite")
       .partitionBy("capture_date")        # for partition pruning in section 6
       .parquet(str(SILVER)))
t_silver_write = time.perf_counter() - t0

silver_bytes = sum(f.stat().st_size for f in SILVER.rglob("*.parquet"))
print(f"silver written in {t_silver_write:.1f} s, {silver_bytes/1024**3:.2f} GB")
print(f"{len(list(SILVER.glob('capture_date=*')))} daily partitions")

defects = dict(D1_header_rows=int(d1_count), D5_duplicates=int(d5),
               D6_inf_nan_columns=len(d6), D6_inf_nan_cells=int(sum(d6.values())),
               D7_negative_duration=int(d7), D7a_out_of_window=int(d7a),
               bronze_rows=int(bronze_rows), silver_rows=int(rows_before))

D5 duplicates: 4,243,106 (26.14%)
silver written in 541.0 s, 1.88 GB
10 daily partitions


## 5. M1 - storage formats

In [ ]:
# 5.1 footprints
csv_bytes = sum(f.stat().st_size for f in CSV_FILES)

m1 = pd.DataFrame([
    dict(format="CSV raw",                  gb=csv_bytes/1024**3,    write_s=None),
    dict(format="Parquet bronze",           gb=bronze_bytes/1024**3, write_s=round(t_bronze_write,1)),
    dict(format="Parquet silver partitioned", gb=silver_bytes/1024**3, write_s=round(t_silver_write,1)),
])
m1["vs_csv"] = ((csv_bytes/1024**3) / m1["gb"]).round(2).astype(str) + "x"
m1["gb"] = m1["gb"].round(2)
display(m1)

print(f"\nParquet holds the same {bronze_rows:,} flows in "
      f"{bronze_bytes/csv_bytes:.1%} of the CSV size.")

,format,gb,write_s,vs_csv
0,CSV raw,6.41,NaN,1.0x
1,Parquet bronze,2.39,441.4,2.68x
2,Parquet silver partitioned,1.88,541.0,3.42x



Parquet holds the same 16,233,002 flows in 37.2% of the CSV size.


## 6. M2 - read performance

Three query shapes on CSV vs partitioned Parquet. Gap should widen from the
first to the third as projection and partition pruning kick in.

In [ ]:
# 6.1 timings
def timed(fn, repeats=2):
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

csv_src = spark.read.csv(str(DATA / "*.csv"), header=True, inferSchema=False)
pq_src  = spark.read.parquet(str(SILVER))
one_day = "2018-03-02"                     # the Bot day

queries = {
    "row count (parquet: footer metadata only)": (
        lambda: csv_src.count(),
        lambda: pq_src.count()),
    "single column max (projection, 1 col of 80)": (
        lambda: csv_src.agg(F.max("Flow Duration")).collect(),
        lambda: pq_src.agg(F.max("Flow Duration")).collect()),
    "filter one day (partition pruning)": (
        lambda: csv_src.filter(F.col("Timestamp").contains("02/03/2018")).count(),
        lambda: pq_src.filter(F.col("capture_date") == one_day).count()),
    "filter + mean on non-partition col (full scan both)": (
        lambda: csv_src.filter(F.col("Protocol") == "6")
                       .agg(F.avg(F.col("Flow Duration").cast("double"))).collect(),
        lambda: pq_src.filter(F.col("Protocol") == 6)
                      .agg(F.avg("Flow Duration")).collect()),
}

rows = []
for name, (q_csv, q_pq) in queries.items():
    t_csv, t_pq = timed(q_csv), timed(q_pq)
    rows.append(dict(query=name, csv_s=round(t_csv, 2), parquet_s=round(t_pq, 2),
                     speedup=f"{t_csv/t_pq:.1f}x"))
    print(f"{t_csv/t_pq:8.1f}x  CSV {t_csv:7.2f}s | Parquet {t_pq:6.2f}s | {name}")

m2 = pd.DataFrame(rows)
print("\nThe last row is the fair comparison: both engines read and compute.")
print("The first three isolate metadata access, projection and partition pruning.")

   323.3x  CSV  322.10s | Parquet   1.00s | row count (parquet: footer metadata only)
   383.9x  CSV  319.36s | Parquet   0.83s | single column max (projection, 1 col of 80)
  2770.9x  CSV  321.57s | Parquet   0.12s | filter one day (partition pruning)
   217.4x  CSV  318.31s | Parquet   1.46s | filter + mean on non-partition col (full scan both)

The last row is the fair comparison: both engines read and compute.
The first three isolate metadata access, projection and partition pruning.


## 7. Gold - one table per protocol

| | split | leaky features | duplicates | classes |
|---|---|---|---|---|
| P1 | random 70/15/15 | kept | kept | all |
| P2 | within-day temporal | dropped | removed | all |
| P3 | as P2 | dropped | removed | Bot test only |

Three separate tables means the protocol is a property of the data, not a branch
in the model code, so notebook 02 trains identical models against identical inputs.

Split is within-day, not by day. Every attack family here sits on only one or two
days, so a by-day split would put Infilteration and Bot in test having never been
trained on, which collapses P2 into P3.

In [ ]:
# 7.1 per-day stratified sample
# Big data sections above used all 16.2M rows. Training can't - CPU only torch,
# nine models. Cap is per (day, class) so one big day can't define a class, and
# the same sample feeds all nine cells.
CAP_PER_GROUP = 40_000

silver_df = spark.read.parquet(str(SILVER))
ordered = Window.partitionBy("capture_date", "Label").orderBy(F.rand(seed=42))

sampled = (silver_df
           .withColumn("_rn", F.row_number().over(ordered))
           .filter(F.col("_rn") <= CAP_PER_GROUP)
           .drop("_rn"))

n_sample = sampled.count()
print(f"sample: {n_sample:,} rows ({100*n_sample/rows_before:.1f}% of silver)\n")

dist = sampled.groupBy("Label").count().orderBy(F.desc("count")).toPandas()
dist["pct"] = (100*dist["count"]/n_sample).round(3)
display(dist)

sample: 813,648 rows (5.0% of silver)



,Label,count,pct
0,Benign,400000,49.161
1,Infilteration,80000,9.832
2,SSH-Bruteforce,40000,4.916
3,DoS attacks-GoldenEye,40000,4.916
4,DoS attacks-Hulk,40000,4.916
5,DoS attacks-SlowHTTPTest,40000,4.916
6,Bot,40000,4.916
7,DDOS attack-HOIC,40000,4.916
8,DDoS attacks-LOIC-HTTP,40000,4.916
9,FTP-BruteForce,40000,4.916


In [ ]:
# 7.2 P1 - random split, leaky features kept
# Timestamp goes to epoch seconds rather than being dropped. As a string a net
# can't use it, so the leak P1 exists to expose would never actually fire.
p1 = (sampled.withColumn("ts_epoch", F.col("ts").cast("long"))
             .drop("ts", "capture_date", "Timestamp"))
p1_train, p1_val, p1_test = p1.randomSplit([0.70, 0.15, 0.15], seed=42)

for name, d in [("train", p1_train), ("val", p1_val), ("test", p1_test)]:
    d.write.mode("overwrite").parquet(str(GOLD / "p1_naive" / name))
print(f"P1: train {p1_train.count():,} val {p1_val.count():,} test {p1_test.count():,}")

P1: train 570,003 val 121,860 test 121,785


In [ ]:
# 7.3 P2 - within-day temporal split, leaky features dropped
p2_base = sampled.dropDuplicates(
    [c for c in sampled.columns if c not in ("ts", "capture_date")])

day_order = Window.partitionBy("capture_date", "Label").orderBy("ts")
p2_ranked = p2_base.withColumn("_pct", F.percent_rank().over(day_order))

p2_train = p2_ranked.filter(F.col("_pct") <  0.70)
p2_val   = p2_ranked.filter((F.col("_pct") >= 0.70) & (F.col("_pct") < 0.85))
p2_test  = p2_ranked.filter(F.col("_pct") >= 0.85)

drop_cols = [c for c in LEAKY if c in p2_ranked.columns] + ["_pct", "ts", "capture_date"]
for name, d in [("train", p2_train), ("val", p2_val), ("test", p2_test)]:
    d.drop(*drop_cols).write.mode("overwrite").parquet(str(GOLD / "p2_corrected" / name))

print(f"P2: train {p2_train.count():,} val {p2_val.count():,} test {p2_test.count():,}")
print(f"dropped: {[c for c in LEAKY if c in p2_ranked.columns]}")

P2: train 528,602 val 114,133 test 111,968
dropped: ['Dst Port', 'Timestamp']


In [ ]:
# 7.3b check every class survived the split
# Read the splits back from disk. p2_train/val/test are lazy - touching them
# re-runs the sampling window
# attacks come in bursts, so a time quantile cut can strand a rare class
chk = []
for name in ("train", "val", "test"):
    d = spark.read.parquet(str(GOLD / "p2_corrected" / name))
    chk.append(d.groupBy("Label").count().toPandas()
                .set_index("Label")["count"].rename(name))

coverage = pd.concat(chk, axis=1).fillna(0).astype(int)
coverage["in_all_3"] = (coverage > 0).all(axis=1)
display(coverage.sort_values("train", ascending=False))

missing = coverage.index[~coverage["in_all_3"]].tolist()
if missing:
    print(f"\nnot in all three partitions: {missing}")
    print("exclude these from the macro average, in P1 as well as P2,")
    print("so the protocols are scored over the same class set")
else:
    print("\nall classes present in all three partitions")

# save for notebook 02 so the macro average uses the same class set
EVAL_CLASSES = sorted(coverage.index[coverage["in_all_3"]].tolist())
(RESULTS / "eval_classes.json").write_text(json.dumps(
    dict(eval_classes=EVAL_CLASSES, excluded=missing), indent=2))
print(f"\n{len(EVAL_CLASSES)} classes usable for macro-F1, written to results/eval_classes.json")


,train,val,test,in_all_3
Label,,,,
Benign,280039,60144,59758,True
Infilteration,56003,12781,11205,True
DDoS attacks-LOIC-HTTP,28010,5996,5994,True
DoS attacks-Hulk,27985,5939,5816,True
DDOS attack-HOIC,27982,5982,5986,True
DoS attacks-GoldenEye,27980,6029,5942,True
Bot,27948,5988,5987,True
SSH-Bruteforce,22373,4795,4793,True
FTP-BruteForce,12802,2742,2742,True



all classes present in all three partitions

15 classes usable for macro-F1, written to results/eval_classes.json


In [ ]:
# 7.4 P3 - Bot removed from train and val
# Also reads from disk for the same reason.
p2_tr = spark.read.parquet(str(GOLD / "p2_corrected" / "train"))
p2_va = spark.read.parquet(str(GOLD / "p2_corrected" / "val"))
p2_te = spark.read.parquet(str(GOLD / "p2_corrected" / "test"))

p3_train = p2_tr.filter(F.col("Label") != ZERO_DAY)
p3_val   = p2_va.filter(F.col("Label") != ZERO_DAY)
p3_test  = p2_te                                      # test keeps Bot

for name, d in [("train", p3_train), ("val", p3_val), ("test", p3_test)]:
    d.write.mode("overwrite").parquet(str(GOLD / "p3_zeroday" / name))

# verify from what was actually written, not from the lazy plan
w_tr = spark.read.parquet(str(GOLD / "p3_zeroday" / "train"))
w_te = spark.read.parquet(str(GOLD / "p3_zeroday" / "test"))
bot_train = w_tr.filter(F.col("Label") == ZERO_DAY).count()
bot_test  = w_te.filter(F.col("Label") == ZERO_DAY).count()

print(f"P3: train {w_tr.count():,} val "
      f"{spark.read.parquet(str(GOLD / 'p3_zeroday' / 'val')).count():,} "
      f"test {w_te.count():,}")
print(f"{ZERO_DAY} in train: {bot_train} (must be 0), in test: {bot_test:,}")
assert bot_train == 0, "zero-day class leaked into training"
print("assertion passed")

P3: train 500,654 val 108,145 test 111,968
Bot in train: 0 (must be 0), in test: 5,987
assertion passed


## 8. Summary

In [ ]:
summary = dict(
    dataset=dict(files=len(CSV_FILES), csv_gb=round(csv_bytes/1024**3, 2),
                 bronze_rows=int(bronze_rows), silver_rows=int(rows_before),
                 sample_rows=int(n_sample), cap_per_group=CAP_PER_GROUP),
    legacy_baseline=legacy_projection,
    defects=defects,
    m1_storage=m1.to_dict("records"),
    m2_read_performance=m2.to_dict("records"),
    spark=dict(version=spark.version, cores=spark.sparkContext.defaultParallelism),
)
(RESULTS / "nb01_measurements.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "dataset": {
    "files": 10,
    "csv_gb": 6.41,
    "bronze_rows": 16233002,
    "silver_rows": 16232929,
    "sample_rows": 813648,
    "cap_per_group": 40000
  },
  "legacy_baseline": {
    "disk_gb": 6.41,
    "projected_mem_gb": 94.6,
    "projected_load_min": 9.3,
    "machine_ram_gb": 7.9,
    "available_gb": 3.2,
    "single_file_rows": 331125,
    "memory_multiplier": 14.8,
    "rows_loaded_single_node": 5245725,
    "pct_of_corpus": 32.3,
    "stopped_at": "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv after 5,245,725 rows",
    "elapsed_min": 2.0,
    "chunk_rows": 100000
  },
  "defects": {
    "D1_header_rows": 59,
    "D5_duplicates": 4243106,
    "D6_inf_nan_columns": 2,
    "D6_inf_nan_cells": 191520,
    "D7_negative_duration": 0,
    "D7a_out_of_window": 14,
    "bronze_rows": 16233002,
    "silver_rows": 16232929
  },
  "m1_storage": [
    {
      "format": "CSV raw",
      "gb": 6.41,
      "write_s": NaN,
      "vs_csv": "1.0x"
    },
    {
      "format":

What this established:

1. pandas limit measured, not assumed
2. two defects here fail silently - D2 schema mismatch and the date format trap
3. defect counts quantified
4. storage and read performance measured
5. three gold tables encode the three protocols

In [ ]:
spark.stop()